# Toy Membership Inference Attack (MIA)

This notebook demonstrates a simple **membership inference** attack:

- Train a model that **overfits**.
- Compute a **score** per sample (here: per-sample loss).
- Decide **member vs non-member** with a threshold rule.

This is a toy setting to build intuition; real attacks can use shadow models and richer signals.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def make_rings(n, noise=0.08, r0=1.0, r1=2.0):
    """Binary classification rings dataset.
    Returns X: (n, 2), y: (n, 1)
    """
    n0 = n // 2
    n1 = n - n0

    t0 = 2 * np.pi * np.random.rand(n0).astype(np.float32)
    t1 = 2 * np.pi * np.random.rand(n1).astype(np.float32)

    x0 = np.stack([r0 * np.cos(t0), r0 * np.sin(t0)], axis=1).astype(np.float32)
    x1 = np.stack([r1 * np.cos(t1), r1 * np.sin(t1)], axis=1).astype(np.float32)

    x0 += noise * np.random.randn(n0, 2).astype(np.float32)
    x1 += noise * np.random.randn(n1, 2).astype(np.float32)

    X = np.concatenate([x0, x1], axis=0)  # (n, 2)
    y = np.concatenate([
        np.zeros((n0, 1), dtype=np.float32),
        np.ones((n1, 1), dtype=np.float32),
    ], axis=0)  # (n, 1)

    perm = np.random.permutation(n)
    return X[perm], y[perm]


N_train = 200
N_test = 2500

X_train_np, y_train_np = make_rings(N_train, noise=0.10)
X_test_np, y_test_np = make_rings(N_test, noise=0.10)

# Tensors: X_train: (N_train, 2), y_train: (N_train, 1)
#          X_test:  (N_test,  2), y_test:  (N_test,  1)
X_train = torch.tensor(X_train_np, device=device)
y_train = torch.tensor(y_train_np, device=device)
X_test = torch.tensor(X_test_np, device=device)
y_test = torch.tensor(y_test_np, device=device)

X_train.shape, X_test.shape

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (N, 2) -> logits: (N, 1)
        return self.net(x)


model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

def acc(model, X_eval, y_eval):
    # X_eval: (N, 2), y_eval: (N, 1)
    with torch.no_grad():
        probs = torch.sigmoid(model(X_eval))  # (N, 1)
        preds = (probs >= 0.5).float()  # (N, 1)
    return float((preds == y_eval).float().mean().item())


In [ ]:
# Train long enough to create an overfitting gap (toy setting)
epochs = 2000
for e in range(epochs):
    logits = model(X_train)  # (N_train, 1)
    loss = loss_fn(logits, y_train)  # scalar
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if (e + 1) % 400 == 0:
        a_tr = acc(model, X_train, y_train)
        a_te = acc(model, X_test, y_test)
        print(f"epoch={e+1:4d}  loss={loss.item():.4f}  train_acc={a_tr:.3f}  test_acc={a_te:.3f}")

train_acc = acc(model, X_train, y_train)
test_acc = acc(model, X_test, y_test)
print(f"Final: train_acc={train_acc:.3f}, test_acc={test_acc:.3f}")

In [ ]:
# Per-sample loss as the MIA score.
# reduction='none' gives a loss per example.
loss_per = nn.BCEWithLogitsLoss(reduction="none")

with torch.no_grad():
    # logits_train: (N_train, 1), losses_train: (N_train, 1)
    logits_train = model(X_train)
    losses_train = loss_per(logits_train, y_train)

    # logits_test: (N_test, 1), losses_test: (N_test, 1)
    logits_test = model(X_test)
    losses_test = loss_per(logits_test, y_test)

# Flatten to vectors for thresholding: (N_train,), (N_test,)
s_train = losses_train.view(-1).detach().cpu().numpy()
s_test = losses_test.view(-1).detach().cpu().numpy()

print(s_train.shape, s_test.shape, f"mean_loss_train={s_train.mean():.4f}", f"mean_loss_test={s_test.mean():.4f}")

In [ ]:
# Attack rule: predict member if loss <= tau
# Build a balanced evaluation set (same number of train/test examples)
m = min(len(s_train), len(s_test))
s_member = s_train[:m]
s_non = s_test[:m]

scores = np.concatenate([s_member, s_non], axis=0)  # (2m,)
labels = np.concatenate([np.ones(m), np.zeros(m)], axis=0)  # (2m,) 1=member, 0=non

# Candidate thresholds from observed scores
taus = np.quantile(scores, np.linspace(0.02, 0.98, 200))

best = None
for tau in taus:
    pred = (scores <= tau).astype(np.float32)  # (2m,)
    acc_attack = (pred == labels).mean()
    tpr = (pred[labels == 1] == 1).mean()
    fpr = (pred[labels == 0] == 1).mean()
    bal = 0.5 * (tpr + (1.0 - fpr))
    if best is None or bal > best["bal_acc"]:
        best = {"tau": float(tau), "acc": float(acc_attack), "tpr": float(tpr), "fpr": float(fpr), "bal_acc": float(bal)}

best

In [ ]:
tau = best["tau"]

plt.figure(figsize=(8, 4))
plt.hist(s_non, bins=40, alpha=0.6, label="non-member (test)")
plt.hist(s_member, bins=40, alpha=0.6, label="member (train)")
plt.axvline(tau, color="white", linestyle="--", linewidth=2, label=f"tau={tau:.3f}")
plt.title("Membership inference score distributions (loss)")
plt.xlabel("per-sample loss (lower => more likely member)")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Attack results (balanced eval): bal_acc={best['bal_acc']:.3f}, acc={best['acc']:.3f}, TPR={best['tpr']:.3f}, FPR={best['fpr']:.3f}")
print("Interpretation: a larger generalization gap -> easier membership inference.")